# 6. Deploy the Agent

**Goal:** Package and deploy the forecasting agent as a DataRobot **Agentic Workflow** so it can run in a serverless Prediction Environment and be governed in a Use Case.

**Key Concept:**
We turn the agent into a deployable “model” using **DRUM** entrypoints and artifact packaging. This notebook:
- Writes `agent.py` (a `create_agent()` factory)
- Writes `custom.py` (DRUM `load_model()` + OpenAI-compatible `chat()`)
- Bundles everything in `agent_artifacts/`
- Uploads as a Custom Model Version + builds dependencies
- Registers a Model Package
- Creates a Deployment and polls until it becomes active

Runtime behavior (deployment IDs, dataset IDs, LLM model) is driven by runtime parameters / env vars—not hardcoded values.

## Step 1: Write `agent.py` (agent factory)

**Goal:** Provide a `create_agent()` factory for DRUM to load at runtime.

**Key Concept:**
Do **not** instantiate the agent at import time. Import-time failures are painful in serverless; a factory keeps startup predictable and makes failures easier to debug.

In [ ]:
%%writefile agent_artifacts/agent.py
import os
import datarobot as dr
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.mcp import MCPServerStreamableHTTP


def create_agent():
    """
    Factory that builds the agent and the MCP server connection.
    """
    mcp_deployment_id = os.environ.get("MCP_DEPLOYMENT_ID")
    llm_model_id = os.environ.get("MODEL_NAME")

    dr_client = dr.Client()
    llmgw_base_url = dr_client.endpoint.rstrip("/") + "/genai/llmgw"

    # Configure the Model
    provider = OpenAIProvider(
        api_key=dr_client.token,
        base_url=llmgw_base_url,
    )
    model = OpenAIChatModel(llm_model_id, provider=provider)

    # Configure MCP Server (if deployment ID is set)
    server = None
    if mcp_deployment_id:
        server = MCPServerStreamableHTTP(
            f"{dr_client.endpoint}/deployments/{mcp_deployment_id}/directAccess/mcp",
            headers={
                "Authorization": f"Bearer {dr_client.token}",
                "x-datarobot-api-token": dr_client.token,
            },
            timeout=60.0,
        )

    # Define the Agent
    if server:
        agent = Agent(
            model=model,
            toolsets=[server],
            system_prompt=(
                "You are a helpful forecasting assistant. "
                "Use the available forecasting tools provided via MCP to predict future values."
            ),
        )
    else:
        agent = Agent(
            model=model,
            system_prompt="You are a helpful assistant.",
        )

    return agent, server

In [ ]:
import os

# (Optional) Environment sanity-check
# This is useful to confirm which Use Case and DataRobot settings are available in the notebook runtime.
for key, value in os.environ.items():
    if "DATAROBOT" in key.upper() or "USE_CASE" in key.upper():
        print(f"{key}={value}")

current_use_case = os.getenv("DATAROBOT_DEFAULT_USE_CASE")
print("DATAROBOT_DEFAULT_USE_CASE:", current_use_case)

## Step 2: Write `custom.py` (DRUM entrypoint)

**Goal:** Implement DRUM hooks so the agent can be deployed as an Agentic Workflow.

**Key Concept:**
Defining `load_model()` prevents DRUM from trying to auto-detect a traditional model artifact (`.pkl`, `.onnx`, etc.). We also implement an OpenAI-compatible `chat()` handler that forwards requests to the agent.

In [ ]:
%%writefile agent_artifacts/custom.py
import asyncio
import concurrent.futures
import time
import traceback
from typing import Any

from openai.types.chat import ChatCompletion, ChatCompletionMessage
from openai.types.chat.chat_completion import Choice

from agent import create_agent


def load_model(code_dir: str) -> Any:
    agent, server = create_agent()
    return {"agent": agent, "server": server}


def _run_sync(coro):
    """Run an async coroutine, handling cases where an event loop may already exist."""
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None

    if loop and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor() as pool:
            return pool.submit(asyncio.run, coro).result()
    else:
        return asyncio.run(coro)


async def _run_agent(agent, server, prompt: str) -> str:
    try:
        if server:
            async with server:
                result = await agent.run(prompt)
                return result.output
        else:
            result = await agent.run(prompt)
            return result.output
    except BaseException as e:
        return f"Agent error:\n{traceback.format_exc()}"


def chat(completion_create_params, model, **kwargs):
    messages = completion_create_params.get("messages", [])

    prompt_parts = []
    for msg in messages:
        role = msg.get("role", "")
        content = msg.get("content", "")
        if role == "system":
            prompt_parts.insert(0, f"System: {content}")
        elif role == "user":
            prompt_parts.append(content)

    prompt = "\n".join(prompt_parts) if prompt_parts else "Hello"

    agent = model["agent"]
    server = model["server"]
    response_text = _run_sync(_run_agent(agent, server, prompt))

    return ChatCompletion(
        id="chatcmpl-custom",
        created=int(time.time()),
        model=completion_create_params.get("model", "datarobot-deployed-llm"),
        object="chat.completion",
        choices=[
            Choice(
                index=0,
                message=ChatCompletionMessage(
                    role="assistant",
                    content=response_text,
                ),
                finish_reason="stop",
            )
        ],
    )

## Step 3: Create minimal project files (deployment hygiene)

**Goal:** Add `pyproject.toml`, `requirements.txt`, and a lockfile so the build system has explicit metadata.

**Key Concept:**
These files help avoid packaging/startup warnings and make dependency builds more deterministic.

In [ ]:
from pathlib import Path
import subprocess

# Step 3.1 Write minimal project metadata into agent_artifacts/
# This helps dependency builds and reduces startup warnings.
def create_project_files():
    Path("agent_artifacts").mkdir(exist_ok=True)

    # Write pyproject.toml
    Path("agent_artifacts/pyproject.toml").write_text(
        """\
[project]
name = "Ercot-agent"
version = "0.0.1"
dependencies = [
    "datarobot",
    "pydantic",
    "pydantic-ai",
]
"""
    )

    # Write requirements.txt
    Path("agent_artifacts/requirements.txt").write_text(
        """\
datarobot
pydantic
pydantic-ai
"""
    )

    # Generate uv.lock (optional; requires uv to be installed)
    subprocess.run(["uv", "lock"], cwd="agent_artifacts")


create_project_files()
                                                                                                                             

## Step 4: Verify the artifact folder before uploading

**Goal:** Confirm `agent_artifacts/` contains all required entrypoints and dependency files.

**Key Concept:**
If any of these files are missing, the serverless build/deploy will fail later (and it’s harder to debug).

In [ ]:
from pathlib import Path

ARTIFACT_DIR = Path("agent_artifacts").resolve()
print("Artifact dir:", ARTIFACT_DIR)
print("Exists:", ARTIFACT_DIR.exists())

for name in ["requirements.txt","agent.py", "custom.py", "pyproject.toml", "uv.lock"]:
    p = ARTIFACT_DIR / name
    print(f"{name}: {'✅' if p.exists() else '❌'} {p}")


## Step 5: Create model + version + dependency build + register (Model Package)

**Goal:** Upload the agent artifacts as a Custom Model Version, build the dependency image, and register it into the Model Registry.

**Key Concept:**
This turns the agent into a governed, deployable asset (a Model Package) that can be deployed to a serverless Prediction Environment.

In [ ]:
import time
from pathlib import Path
import datarobot as dr

dr_client = dr.Client()

# -----------------------------
# CONFIG
# -----------------------------
AGENT_NAME = "SKO Ercot Forecast Agent (Agentic Workflow)"
REGISTERED_MODEL_NAME = "SKO Ercot Forecast Agent"
TARGET_TYPE = dr.enums.TARGET_TYPE.AGENTIC_WORKFLOW

ARTIFACT_DIR = Path("agent_artifacts").resolve()

# -----------------------------
# SAFETY CHECKS (entrypoint)
# -----------------------------
required = ["custom.py", "agent.py", "requirements.txt"]
missing = [f for f in required if not (ARTIFACT_DIR / f).exists()]
if missing:
    raise RuntimeError(f"Missing required artifact files in {ARTIFACT_DIR}: {missing}")

print("✅ Artifacts OK:", ARTIFACT_DIR)
for f in required:
    print(" -", f)

# -----------------------------
# 1) Execution Environment
# -----------------------------
envs = dr.ExecutionEnvironment.list(search_for="GenAI")
genai_env = next((e for e in envs if "python" in e.name.lower()), None)
if not genai_env:
    raise ValueError("Could not find a suitable Python GenAI Execution Environment!")
print(f"🚀 Execution Environment: {genai_env.name} (id={genai_env.id})")

# -----------------------------
# 2) Create / Reuse Custom Inference Model
# -----------------------------
try:
    custom_model = dr.CustomInferenceModel.create(
        name=AGENT_NAME,
        target_type=TARGET_TYPE,
        target_name="response",
        description="Serverless Agentic Workflow (DRUM custom.py entrypoint).",
        language="python",
    )
    print(f"✅ Created Custom Inference Model: {custom_model.name} (id={custom_model.id})")
except dr.errors.ClientError:
    custom_model = next(m for m in dr.CustomInferenceModel.list() if m.name == AGENT_NAME)
    print(f"✅ Using existing Custom Inference Model: {custom_model.name} (id={custom_model.id})")

# -----------------------------
# 3) Create Custom Model Version (upload artifacts)
# -----------------------------
print("📤 Creating custom model version (upload artifacts)...")
version = dr.CustomModelVersion.create_clean(
    custom_model_id=custom_model.id,
    base_environment_id=genai_env.id,
    folder_path=str(ARTIFACT_DIR),
)
print(f"✅ Custom Model Version Created: {version.id}")

# -----------------------------
# 4) Dependency Image Build (required in your tenant)
# -----------------------------
def start_dependency_build(custom_model_id: str, custom_model_version_id: str):
    print("🔨 Starting dependency image build...")
    try:
        dr.CustomModelVersionDependencyBuild.start_build(
            custom_model_id=custom_model_id,
            custom_model_version_id=custom_model_version_id,
        )
        print("✅ Dependency build trigger sent.")
    except dr.errors.ClientError as e:
        msg = str(e).lower()
        if "in progress" in msg or "already" in msg:
            print("ℹ️ Dependency build already running.")
        else:
            raise

def wait_for_dependency_build(custom_model_id: str, custom_model_version_id: str, timeout_s: int = 1800, poll_s: int = 10):
    print("⏳ Waiting for dependency image build to finish...")
    start = time.time()
    last = None
    not_started_retries = 0

    while True:
        try:
            info = dr.CustomModelVersionDependencyBuild.get_build_info(
                custom_model_id=custom_model_id,
                custom_model_version_id=custom_model_version_id,
            )
        except dr.errors.ClientError as e:
            # brief eventual consistency right after triggering
            if "not been started" in str(e).lower() and not_started_retries < 12:
                not_started_retries += 1
                print(f"   build info not ready... ({not_started_retries}/12)", end="\r")
                time.sleep(5)
                continue
            raise

        status = getattr(info, "build_status", None) or getattr(info, "status", None) or "unknown"
        if status != last:
            print(f"   dependency build status → {status}")
            last = status

        s = str(status).strip().lower()
        if s in {"success", "succeeded", "complete", "completed"}:
            print("✅ Dependency image build complete.")
            return info
        if s in {"failed", "error"}:
            raise RuntimeError("❌ Dependency image build failed. Check Build Logs in the DataRobot UI.")
        if int(time.time() - start) > timeout_s:
            raise TimeoutError(f"Timed out waiting for dependency image build. Last status={status}")

        time.sleep(poll_s)

start_dependency_build(custom_model.id, version.id)
wait_for_dependency_build(custom_model.id, version.id)

# -----------------------------
# 5) Register -> Model Package (Model Registry)
# -----------------------------
print("📦 Registering custom model version into Model Registry (creates model package)...")
rmv = dr.RegisteredModelVersion.create_for_custom_model_version(
    custom_model_version_id=version.id,
    registered_model_name=REGISTERED_MODEL_NAME,
    name=f"{REGISTERED_MODEL_NAME} - {time.strftime('%Y%m%d_%H%M%S')}",
)
model_package_id = rmv.id
print(f"✅ model_package_id: {model_package_id}")

# (Optional) show current model package buildStatus immediately
mp = dr_client.get(f"modelPackages/{model_package_id}/").json()
print("📦 model package buildStatus (initial):", mp.get("buildStatus"))


# -----------------------------
# Link Registered model to the use case
# -----------------------------  
use_case_id = os.getenv("DATAROBOT_DEFAULT_USE_CASE")  # or hardcode                                                                                                                                          
#registered_model_version_id = "your_registered_model_version_id"                                                                                                                                         
                                                                                                                                                                                                          
# Make the API call directly                                                                                                                                                                             
response = dr.Client().post(                                                                                                                                                                             
    f"useCases/{use_case_id}/registeredModelVersions/{model_package_id}"                                                                                                                     
)                                                                                                                                                                                                        
                                                                                                                                                                                                           
print(f"🔗 Linked registered model version {model_package_id} to use case {use_case_id}") 


## Step 6: Deploy to a Prediction Environment

**Goal:** Wait for the Model Package build, then create a Deployment in your serverless Prediction Environment.

**Key Concept:**
Serverless deployments can take several minutes while images/build steps complete; polling build status avoids creating a deployment too early.

In [ ]:
import time
import datarobot as dr

dr_client = dr.Client()

# -----------------------------
# CONFIG
# -----------------------------
PREDICTION_ENV_ID = "697ce1c013583b87d7b7c529"  # serverless prediction environment
AGENT_NAME = "SKO Ercot Forecast Agent (Agentic Workflow)"

# -----------------------------
# Helpers
# -----------------------------
def wait_for_model_package_build(mp_id: str, timeout_s: int = 1800, poll_s: int = 10):
    start = time.time()
    last = None
    while True:
        mp = dr_client.get(f"modelPackages/{mp_id}/").json()
        status = mp.get("buildStatus", "unknown")

        if status != last:
            print(f"📦 modelPackages/{mp_id} buildStatus → {status}")
            last = status

        s = str(status).strip().lower()
        if s in {"complete", "completed", "success", "succeeded", "ready"}:
            print("✅ Model package build complete.")
            return mp
        if s in {"failed", "error"}:
            raise RuntimeError(f"❌ Model package build failed. buildStatus={status}")
        if s in {"n/a", "na", "not_applicable"}:
            raise RuntimeError("⚠️ buildStatus is N/A. Re-register the model to trigger build.")
        if int(time.time() - start) > timeout_s:
            raise TimeoutError(f"Timed out waiting for model package build. Last buildStatus={status}")

        time.sleep(poll_s)

def get_deployment_by_label(label: str):
    deps = dr.Deployment.list()
    matches = [d for d in deps if getattr(d, "label", None) == label]
    return matches[0] if matches else None

# -----------------------------
# Verify prediction environment
# -----------------------------
pred_env = dr.PredictionEnvironment.get(PREDICTION_ENV_ID)
print(f"🚀 Prediction Environment: {pred_env.name} (id={pred_env.id}, platform={pred_env.platform})")

# -----------------------------
# Wait for model package build
# -----------------------------
wait_for_model_package_build(model_package_id, timeout_s=1800, poll_s=10)

# -----------------------------
# Delete existing deployment (optional)
# -----------------------------
existing = get_deployment_by_label(AGENT_NAME)
if existing:
    print(f"♻️ Deleting existing deployment '{AGENT_NAME}': {existing.id}")
    existing.delete()
    time.sleep(10)

# -----------------------------
# Create deployment to prediction environment
# -----------------------------
print("🚀 Creating deployment...")
deployment = dr.Deployment.create_from_registered_model_version(
    model_package_id=model_package_id,
    label=AGENT_NAME,
    prediction_environment_id=PREDICTION_ENV_ID,
    max_wait=900,  # serverless can take time
)

print(f"✅ Deployment created: {deployment.id}")
base_url = dr_client.endpoint.replace("/api/v2", "")
print(f"View: {base_url}/console/deployments/{deployment.id}/overview")

                                                                                                                                                                                                    
# -----------------------------
# Assign deployment to the use case
# -----------------------------                                                                                                                                                                                                        
# Get use case ID                                                                                                                                                                                               
use_case_id = os.getenv("DATAROBOT_DEFAULT_USE_CASE")                                                                                                                                         
                                                                                                                                                                                                          
# Make the API call directly                                                                                                                                                                             
response = dr.Client().post(                                                                                                                                                                             
    #f"useCases/{use_case_id}/deployment.id"                                                                                                                     
    f"useCases/{use_case_id}/deployments/{deployment.id}"                                                                                                                                                                                                     
)
print(f"🔗 Linked model deployment {deployment.id} to use case {use_case_id}") 


## Step 7: Validate the deployment (poll until active)

**Goal:** Confirm the deployment reaches an `active/ready` state.

**Key Concept:**
If the deployment fails, the detailed deployment payload often contains a `statusId` you can use to retrieve the underlying async job details for debugging.

In [ ]:
import time
import json
import datarobot as dr

dr_client = dr.Client()

DEPLOYMENT_ID = deployment.id  # from Cell 6

def deployment_detail(deployment_id: str) -> dict:
    return dr_client.get(f"deployments/{deployment_id}/").json()

def get_async_job(status_id: str) -> dict:
    return dr_client.get(f"asyncJobs/{status_id}/").json()

print("🔎 Polling deployment status...")
for i in range(90):  # ~15 minutes at 10s
    d = deployment_detail(DEPLOYMENT_ID)

    status = d.get("status")
    health = d.get("health") or d.get("serviceHealth") or d.get("availability")

    print(f"[{i}] status={status} health={health}")

    s = str(status).strip().lower() if status else ""
    if s in {"active", "ready"}:
        print("✅ Deployment is active.")
        break

    if s in {"error", "failed"}:
        print("❌ Deployment errored. Full deployment detail:")
        print(json.dumps(d, indent=2))

        print("\nIf you have an async statusId (from exception output), fetch it like this:")
        print("job = get_async_job('<statusId>'); print(json.dumps(job, indent=2))")
        break

    time.sleep(10)


In [ ]:
import datarobot as dr
from openai import OpenAI

# Step 8: Invoke the deployed Agentic Workflow via OpenAI-compatible chat

# 1. Connect using the notebook session
try:
    dr_client = dr.Client()
    endpoint = dr_client.endpoint.split("/api/v2")[0]
    api_key = dr_client.token
except Exception as e:
    raise RuntimeError(
        "Connection failed. Make sure you are running this inside a DataRobot Notebook."
    ) from e

# 2. Construct the OpenAI-compatible chat base URL for your deployment
CHAT_API_URL = f"{endpoint}/api/v2/deployments/{deployment.id}/v1"

# 3. Initialize the OpenAI client
client = OpenAI(
    base_url=CHAT_API_URL,
    api_key=api_key,
    _strict_response_validation=False,
)

# 4. Send a prompt
prompt = "What would it take to colonize Mars?"
print(f"Sending prompt to Agent: {prompt}...")

completion = client.chat.completions.create(
    model="datarobot-deployed-llm",
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Be detailed."},
        {"role": "user", "content": prompt},
    ],
    max_tokens=512,
)

print("\n--- AGENT RESPONSE ---")
print(completion.choices[0].message.content)